# 🎬 MovieLens 100K — Exploratory Data Analysis

This notebook explores the MovieLens 100K dataset before training our RBM recommendation model.

**Dataset**: 100,000 ratings (1-5) from 943 users on 1,682 movies.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import download_dataset, load_ratings, load_movies, load_users

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

# Download dataset if needed
download_dataset()
print('Ready!')

## 1. Load & Inspect Data

In [ ]:
ratings = load_ratings()
movies = load_movies()
users = load_users()

print(f'Ratings: {len(ratings):,} rows')
print(f'Movies:  {len(movies):,} unique')
print(f'Users:   {len(users):,} unique')
print(f'\nSparsity: {1 - len(ratings) / (len(users) * len(movies)):.2%}')
print()
ratings.head()

In [ ]:
movies.head(10)

In [ ]:
ratings.describe()

## 2. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating value distribution
ax1 = axes[0]
counts = ratings['rating'].value_counts().sort_index()
colors = sns.color_palette('viridis', 5)
ax1.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Rating')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Ratings')
for i, v in enumerate(counts.values):
    ax1.text(counts.index[i], v + 500, f'{v:,}', ha='center', fontsize=9)

# Average rating per user (histogram)
ax2 = axes[1]
user_avg = ratings.groupby('user_id')['rating'].mean()
ax2.hist(user_avg, bins=30, color=colors[2], edgecolor='white', alpha=0.8)
ax2.axvline(user_avg.mean(), color='red', linestyle='--', label=f'Mean = {user_avg.mean():.2f}')
ax2.set_xlabel('Average Rating')
ax2.set_ylabel('Number of Users')
ax2.set_title('Average Rating per User')
ax2.legend()

plt.tight_layout()
plt.savefig('../static/eda_ratings.png', bbox_inches='tight', transparent=True)
plt.show()

## 3. User & Movie Activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ratings per user
user_counts = ratings.groupby('user_id').size()
axes[0].hist(user_counts, bins=50, color='#6c5ce7', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Number of Users')
axes[0].set_title(f'Ratings per User (median={user_counts.median():.0f})')

# Ratings per movie
movie_counts = ratings.groupby('movie_id').size()
axes[1].hist(movie_counts, bins=50, color='#00cec9', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Movies')
axes[1].set_title(f'Ratings per Movie (median={movie_counts.median():.0f})')

plt.tight_layout()
plt.savefig('../static/eda_activity.png', bbox_inches='tight', transparent=True)
plt.show()

print(f'Users with < 20 ratings:  {(user_counts < 20).sum()}')
print(f'Movies with < 5 ratings:  {(movie_counts < 5).sum()}')
print(f'Movies with 0 ratings:    {len(movies) - len(movie_counts)}')

## 4. Genre Analysis

In [ ]:
# Count movies per genre
all_genres = movies['genres'].str.split(', ').explode()
genre_counts = all_genres.value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
colors_g = sns.color_palette('viridis', len(genre_counts))
bars = ax.barh(genre_counts.index[::-1], genre_counts.values[::-1], color=colors_g[::-1])
ax.set_xlabel('Number of Movies')
ax.set_title('Movies per Genre')

for bar, val in zip(bars, genre_counts.values[::-1]):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../static/eda_genres.png', bbox_inches='tight', transparent=True)
plt.show()

## 5. Average Rating by Genre

In [ ]:
# Merge ratings with movies to get genre-level stats
merged = ratings.merge(movies, on='movie_id')

genre_ratings = []
for genre in genre_counts.index:
    mask = merged['genres'].str.contains(genre, na=False)
    avg = merged.loc[mask, 'rating'].mean()
    genre_ratings.append({'genre': genre, 'avg_rating': avg})

gr_df = pd.DataFrame(genre_ratings).sort_values('avg_rating', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_r = sns.color_palette('RdYlGn', len(gr_df))
ax.barh(gr_df['genre'], gr_df['avg_rating'], color=colors_r)
ax.set_xlabel('Average Rating')
ax.set_title('Average Rating by Genre')
ax.set_xlim(2.5, 4.5)

for i, (_, row) in enumerate(gr_df.iterrows()):
    ax.text(row['avg_rating'] + 0.02, i, f"{row['avg_rating']:.2f}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../static/eda_genre_ratings.png', bbox_inches='tight', transparent=True)
plt.show()

## 6. User Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age distribution
axes[0].hist(users['age'], bins=30, color='#6c5ce7', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('User Age Distribution')

# Gender
gender_c = users['gender'].value_counts()
axes[1].pie(gender_c, labels=['Male', 'Female'], autopct='%1.1f%%',
            colors=['#6c5ce7', '#00cec9'], startangle=90)
axes[1].set_title('Gender Distribution')

# Top occupations
occ = users['occupation'].value_counts().head(10)
axes[2].barh(occ.index[::-1], occ.values[::-1], color=sns.color_palette('viridis', 10)[::-1])
axes[2].set_xlabel('Count')
axes[2].set_title('Top 10 Occupations')

plt.tight_layout()
plt.savefig('../static/eda_demographics.png', bbox_inches='tight', transparent=True)
plt.show()

## 7. Rating Matrix Sparsity Visualization

In [ ]:
from src.data_preprocessing import create_user_movie_matrix

matrix = create_user_movie_matrix(ratings)

# Show a subset (first 100 users x 200 movies)
fig, ax = plt.subplots(figsize=(14, 6))
subset = matrix[:100, :200]
ax.imshow(subset > 0, aspect='auto', cmap='viridis', interpolation='nearest')
ax.set_xlabel('Movies (first 200)')
ax.set_ylabel('Users (first 100)')
ax.set_title(f'Rating Matrix Sparsity — {(matrix > 0).mean():.2%} filled')

plt.tight_layout()
plt.savefig('../static/eda_sparsity.png', bbox_inches='tight', transparent=True)
plt.show()

## 8. Key Insights

| Metric | Value |
|--------|-------|
| Total Ratings | 100,000 |
| Unique Users | 943 |
| Unique Movies | 1,682 |
| Sparsity | ~93.7% |
| Most Common Rating | 4 (stars) |
| Most Popular Genre | Drama |

**Observations:**
- The rating matrix is highly sparse (~94%), making it ideal for RBM-based collaborative filtering
- Rating distribution is left-skewed (users tend to rate movies they like)
- Drama and Comedy dominate the genre distribution
- Film-Noir and War genres have the highest average ratings
- The dataset is male-dominated (74%) with a median age around 30